In [ ]:
# validation_framework.py
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Callable, Dict, Iterable, List, Optional, Tuple, Union, Any
import pandas as pd
import numpy as np
import importlib.util
from pathlib import Path
import inspect
import logging
import re
import uuid

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

# -------------------------
# Types
# -------------------------
RuleResult = Union[pd.Index, pd.Series]  # Index of offending rows or boolean mask
SeriesLike = pd.Series

# -------------------------
# Registry
# -------------------------
_RULE_REGISTRY: Dict[str, Callable[..., "ValidationRule"]] = {}

def register_rule(cls_or_factory: Callable[..., "ValidationRule"]) -> Callable[..., "ValidationRule"]:
    """Register a ValidationRule class or factory in the global registry."""
    name = getattr(cls_or_factory, "__name__", repr(cls_or_factory))
    _RULE_REGISTRY[name] = cls_or_factory
    return cls_or_factory

def get_registered_rules() -> Dict[str, Callable[..., "ValidationRule"]]:
    return dict(_RULE_REGISTRY)

# -------------------------
# Base Rule
# -------------------------
class ValidationRule:
    """Base class for atomic validation rules.

    A rule should implement __call__(series) and return either:
      - pd.Index with offending indices, or
      - boolean pd.Series aligned with the series index (True === failing)
    """
    name: str = "validation_rule"

    def __call__(self, series: SeriesLike) -> RuleResult:
        raise NotImplementedError()

    @property
    def returns_mask(self) -> bool:
        """If True, rule returns boolean Series; if False, returns pd.Index"""
        return False

    def normalize_to_index(self, series: SeriesLike, res: RuleResult) -> pd.Index:
        """Normalize rule result to pd.Index of offending rows."""
        if isinstance(res, pd.Index):
            return res
        if isinstance(res, pd.Series):
            # True means failing
            return res.loc[res.astype(bool)].index
        raise TypeError("Rule returned unsupported type")

    def normalize_to_mask(self, series: SeriesLike, res: RuleResult) -> pd.Series:
        """Normalize rule result to boolean Series aligned with series.index (True == failing)."""
        if isinstance(res, pd.Series):
            return res.reindex(series.index, fill_value=False)
        if isinstance(res, pd.Index):
            mask = pd.Series(False, index=series.index)
            mask.loc[res] = True
            return mask
        raise TypeError("Rule returned unsupported type")

# -------------------------
# Utilities
# -------------------------
def _to_numeric_coerce(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def _is_uuid_string(s: str) -> bool:
    try:
        uuid.UUID(s)
        return True
    except Exception:
        return False

# -------------------------
# Built-in rules (vectorized where possible)
# -------------------------

@register_rule
@dataclass
class IsNumericRule(ValidationRule):
    """Flag rows that are NOT numeric (coercion fails)."""
    name: str = "is_numeric"

    def __call__(self, series: pd.Series) -> pd.Index:
        coerced = _to_numeric_coerce(series)
        # failing = coercion is NaN but original is not NaN (missing values ignored)
        failing = coerced.isna() & ~series.isna()
        return series[failing].index

@register_rule
@dataclass
class NotNumericRule(IsNumericRule):
    """Alias to have consistent naming."""
    name: str = "not_numeric"

@register_rule
@dataclass
class InRangeRule(ValidationRule):
    low: float = 0.0
    high: float = 1_000_000.0
    inclusive: bool = True
    ignore_non_numeric: bool = False
    name: str = "in_range"

    def __call__(self, series: pd.Series) -> pd.Index:
        low, high = sorted((self.low, self.high))
        coerced = _to_numeric_coerce(series)
        numeric_mask = ~coerced.isna()
        if self.inclusive:
            out_mask = numeric_mask & ((coerced < low) | (coerced > high))
        else:
            out_mask = numeric_mask & ((coerced <= low) | (coerced >= high))
        if self.ignore_non_numeric:
            return series[out_mask].index
        else:
            non_numeric = coerced.isna() & ~series.isna()
            # combine out_of_range and non_numeric
            out_idx = np.concatenate([series[out_mask].index.values, series[non_numeric].index.values])
            return pd.Index(out_idx)

@register_rule
@dataclass
class NotIntRule(ValidationRule):
    allow_na: bool = True
    name: str = "not_int"

    def __call__(self, series: pd.Series) -> pd.Index:
        # Fast path: integer dtype
        if pd.api.types.is_integer_dtype(series.dtype):
            if self.allow_na:
                return pd.Index([])  # integers (non-null) are fine; NA are allowed
            else:
                # check NA
                return series[series.isna()].index

        coerced = _to_numeric_coerce(series)
        # is integer-like: finite and equal to rounded integer
        # Use vectorized check: coerced == coerced.astype(np.int64) but floats can overflow
        # safer: check fractional part near zero
        frac = (coerced - np.floor(coerced))
        # handle negative numbers: fractional part computed as coerced % 1
        frac = coerced % 1
        is_int_like = (coerced.notna()) & (np.isfinite(coerced)) & (frac == 0)
        if self.allow_na:
            valid = is_int_like | series.isna()
        else:
            valid = is_int_like
        failing_mask = ~valid
        return series.loc[failing_mask.fillna(True)].index

@register_rule
@dataclass
class IsIntRule(NotIntRule):
    """Return rows that are NOT int (alias)."""
    name: str = "is_int"

@register_rule
@dataclass
class NotFloatRule(ValidationRule):
    allow_na: bool = True
    name: str = "not_float"

    def __call__(self, series: pd.Series) -> pd.Index:
        # Float implies numeric but not necessarily integer
        coerced = _to_numeric_coerce(series)
        is_float_like = coerced.notna() & (~(coerced % 1 == 0))
        if self.allow_na:
            valid = (is_float_like) | series.isna()
        else:
            valid = is_float_like
        failing_mask = ~valid
        return series.loc[failing_mask.fillna(True)].index

@register_rule
@dataclass
class ExactDecimalPlacesRule(ValidationRule):
    dp: int = 2
    use_original_strings: bool = False  # if True, expects input series of original strings
    name: str = "exact_decimal_places"

    def __call__(self, series: pd.Series) -> pd.Index:
        dp = int(self.dp)
        if self.use_original_strings:
            s = series.astype("string")
            has_decimal = s.str.contains(".", regex=False, na=False)
            frac = s.where(has_decimal).str.split(".", n=1, expand=True)[1]
            frac_len = frac.str.len().fillna(0)
            ok = (frac_len == dp) | (~has_decimal & (dp == 0)) | s.isna()
            return series[~ok].index
        else:
            coerced = _to_numeric_coerce(series).dropna()
            # numeric-only; if coerced drops non-numeric originally they are not included
            def _check_val(x: float) -> bool:
                # format with high precision, trim trailing zeros
                txt = f"{x:.20f}".rstrip("0").rstrip(".")
                if "." not in txt:
                    return dp == 0
                return len(txt.split(".", 1)[1]) == dp
            bad = [idx for idx, v in coerced.items() if not _check_val(v)]
            # include non-numeric original values (if original notna but coercion NaN)
            non_numeric = series[pd.to_numeric(series, errors="coerce").isna() & ~series.isna()].index
            return pd.Index(np.concatenate([bad, non_numeric]))

@register_rule
@dataclass
class StringFormatRule(ValidationRule):
    pattern: str = ""
    regex_flags: int = 0
    na_fail: bool = True
    name: str = "string_format"

    def __post_init__(self):
        if self.pattern:
            self._compiled = re.compile(self.pattern, flags=self.regex_flags)
        else:
            self._compiled = None

    def __call__(self, series: pd.Series) -> pd.Index:
        s = series.astype("string")
        if self._compiled is None:
            return pd.Index([])  # no pattern => nothing to fail
        match_mask = s.str.match(self._compiled, na=not self.na_fail)
        # offending = not match
        return series[~match_mask.fillna(False)].index

@register_rule
@dataclass
class InLengthRangeRule(ValidationRule):
    min_len: int = 0
    max_len: int = 2**31 - 1
    name: str = "in_length_range"

    def __call__(self, series: pd.Series) -> pd.Index:
        s = series.astype("string")
        ln = s.str.len().fillna(0)
        failing = (ln < self.min_len) | (ln > self.max_len)
        return series[failing].index

@register_rule
@dataclass
class InValueListRule(ValidationRule):
    values: Iterable[Any] = field(default_factory=list)
    name: str = "in_value_list"

    def __call__(self, series: pd.Series) -> pd.Index:
        vals = set(self.values)
        failing = ~series.isin(vals) & ~series.isna()
        return series[failing].index

@register_rule
@dataclass
class MappingRule(ValidationRule):
    mapping: Dict[Any, Any] = field(default_factory=dict)
    allow_missing: bool = False
    name: str = "mapping"

    def __call__(self, series: pd.Series) -> pd.Index:
        # rows where mapping doesn't have a key and not allowed to be missing
        missing_keys_mask = ~series.isin(self.mapping.keys()) & ~series.isna()
        if self.allow_missing:
            return series[pd.Series(False, index=series.index)].index
        else:
            return series[missing_keys_mask].index

@register_rule
@dataclass
class EnumRule(InValueListRule):
    name: str = "enum"

@register_rule
@dataclass
class SubstringExistsRule(ValidationRule):
    substring: str = ""
    case: bool = True
    na_fail: bool = True
    name: str = "substring_exists"

    def __call__(self, series: pd.Series) -> pd.Index:
        s = series.astype("string")
        contains = s.str.contains(self.substring, case=self.case, na=not self.na_fail, regex=False)
        return series[~contains.fillna(False)].index

@register_rule
@dataclass
class OuterReferenceRule(ValidationRule):
    ref_series: pd.Series = field(default_factory=lambda: pd.Series(dtype="object"))
    na_fail: bool = True
    name: str = "outer_reference_missing"

    def __call__(self, series: pd.Series) -> pd.Index:
        ref_set = set(self.ref_series.dropna().unique())
        mask = series.apply(lambda x: (x in ref_set) if not pd.isna(x) else (self.na_fail))
        return series[~mask].index

@register_rule
@dataclass
class InnerReferenceRule(ValidationRule):
    # A template for validating based on other columns in same DataFrame.
    # Must be constructed with a callable `check_func` that accepts (series, df) and returns boolean mask.
    check_func: Optional[Callable[[pd.Series, pd.DataFrame], pd.Series]] = None
    df: Optional[pd.DataFrame] = None
    name: str = "inner_reference"

    def __call__(self, series: pd.Series) -> pd.Index:
        if self.check_func is None or self.df is None:
            raise ValueError("InnerReferenceRule requires check_func and df to be set.")
        mask = self.check_func(series, self.df)  # True == OK or True == fail? we define True == OK
        failing = ~mask.fillna(False)
        return series[failing].index

@register_rule
@dataclass
class UUIDRule(ValidationRule):
    allow_na: bool = True
    name: str = "uuid_check"

    def __call__(self, series: pd.Series) -> pd.Index:
        s = series.astype("string")
        def ok(v):
            if pd.isna(v):
                return self.allow_na
            try:
                uuid.UUID(str(v))
                return True
            except Exception:
                return False
        mask = s.apply(ok)
        return series[~mask].index

@register_rule
@dataclass
class ObjectIdRule(ValidationRule):
    """Basic MongoDB ObjectId check (24 hex chars)."""
    allow_na: bool = True
    name: str = "objectid_check"

    def __call__(self, series: pd.Series) -> pd.Index:
        s = series.astype("string")
        mask = s.str.match(r"^[0-9a-fA-F]{24}$", na=self.allow_na)
        return series[~mask.fillna(False)].index

@register_rule
@dataclass
class IsBooleanRule(ValidationRule):
    name: str = "is_boolean"

    def __call__(self, series: pd.Series) -> pd.Index:
        mask = series.isin([True, False, "True", "False", "true", "false", 0, 1]) | series.isna()
        failing = ~mask
        return series[failing].index

@register_rule
@dataclass
class BooleanMappingRule(ValidationRule):
    mapping: Dict[Any, bool] = field(default_factory=dict)
    name: str = "boolean_mapping"

    def __call__(self, series: pd.Series) -> pd.Index:
        # Failing rows are those not in mapping keys and not NA
        missing = ~series.isin(self.mapping.keys()) & ~series.isna()
        return series[missing].index

@register_rule
@dataclass
class IsDateRule(ValidationRule):
    format: Optional[str] = None  # if provided, check strict formatting; else use coercion
    name: str = "is_date"

    def __call__(self, series: pd.Series) -> pd.Index:
        if self.format:
            # strict parse using pd.to_datetime with format
            parsed = pd.to_datetime(series, format=self.format, errors="coerce")
            failing = parsed.isna() & ~series.isna()
            return series[failing].index
        else:
            parsed = pd.to_datetime(series, errors="coerce")
            failing = parsed.isna() & ~series.isna()
            return series[failing].index

@register_rule
@dataclass
class DateInRangeRule(ValidationRule):
    low: Optional[pd.Timestamp] = None
    high: Optional[pd.Timestamp] = None
    name: str = "date_in_range"

    def __call__(self, series: pd.Series) -> pd.Index:
        parsed = pd.to_datetime(series, errors="coerce")
        mask_valid = parsed.notna()
        failing = pd.Series(False, index=series.index)
        if self.low is not None:
            failing = failing | (mask_valid & (parsed < pd.to_datetime(self.low)))
        if self.high is not None:
            failing = failing | (mask_valid & (parsed > pd.to_datetime(self.high)))
        # non-numeric original non-na -> also fail
        non_parsed = parsed.isna() & ~series.isna()
        return series[(failing | non_parsed)].index

@register_rule
@dataclass
class IsCategoryRule(ValidationRule):
    name: str = "is_category"

    def __call__(self, series: pd.Series) -> pd.Index:
        if pd.api.types.is_categorical_dtype(series.dtype):
            return pd.Index([])
        else:
            # allow series where every value is in a small set? For now, check dtype only
            return series.index if not pd.api.types.is_categorical_dtype(series.dtype) else pd.Index([])

@register_rule
@dataclass
class CustomRule(ValidationRule):
    """Wrap a user-supplied callable that returns a boolean mask or index."""
    func: Callable[[pd.Series], RuleResult] = lambda s: pd.Index([])
    name: str = "custom"

    def __call__(self, series: pd.Series) -> RuleResult:
        return self.func(series)

# -------------------------
# Base Validator (single column)
# -------------------------
@dataclass
class BaseValidator:
    series: pd.Series
    rules: List[ValidationRule] = field(default_factory=list)
    return_mask: bool = False  # if True, return boolean mask per rule; else pd.Index

    def add_rule(self, rule: ValidationRule) -> None:
        if not isinstance(rule, ValidationRule):
            raise TypeError("rule must be ValidationRule instance")
        self.rules.append(rule)

    def add_rule_by_name(self, rule_name: str, *args, **kwargs) -> None:
        factory = _RULE_REGISTRY.get(rule_name)
        if factory is None:
            raise KeyError(f"Rule {rule_name} not found in registry.")
        # factory might be a dataclass class, so instantiate:
        instance = factory(*args, **kwargs)
        if not isinstance(instance, ValidationRule):
            raise TypeError("Registered item did not produce a ValidationRule instance")
        self.add_rule(instance)

    def run(self) -> Dict[str, RuleResult]:
        results: Dict[str, RuleResult] = {}
        for r in self.rules:
            try:
                res = r(self.series)
                if self.return_mask:
                    mask = r.normalize_to_mask(self.series, res)
                    results[r.name] = mask
                else:
                    idx = r.normalize_to_index(self.series, res)
                    results[r.name] = idx
            except Exception as exc:
                logger.exception("Rule %s failed: %s", r.name, exc)
                results[r.name] = pd.Index([]) if not self.return_mask else pd.Series(False, index=self.series.index)
        return results

# -------------------------
# Domain Validators: composed rules
# -------------------------
@dataclass
class NumberValidator:
    series: pd.Series
    rules: List[ValidationRule] = field(default_factory=list)

    def __post_init__(self):
        # default numeric rules; user can customize by passing explicit rules
        if not self.rules:
            self.rules = [
                NotNumericRule(name="not_numeric"),
                NotIntRule(allow_na=True, name="not_int"),
                InRangeRule(name="in_range"),
            ]

    def add_rule(self, rule: ValidationRule):
        self.rules.append(rule)

    def run(self, return_mask: bool = False) -> Dict[str, RuleResult]:
        validator = BaseValidator(self.series, self.rules, return_mask=return_mask)
        return validator.run()

@dataclass
class StringValidator:
    series: pd.Series
    rules: List[ValidationRule] = field(default_factory=list)

    def __post_init__(self):
        if not self.rules:
            # basic: non-empty string check can be custom; leaving defaults empty
            self.rules = []

    def add_rule(self, rule: ValidationRule):
        self.rules.append(rule)

    def run(self, return_mask: bool = False) -> Dict[str, RuleResult]:
        validator = BaseValidator(self.series, self.rules, return_mask=return_mask)
        return validator.run()

@dataclass
class BooleanValidator:
    series: pd.Series
    rules: List[ValidationRule] = field(default_factory=list)

    def __post_init__(self):
        if not self.rules:
            self.rules = [IsBooleanRule()]

    def add_rule(self, rule: ValidationRule):
        self.rules.append(rule)

    def run(self, return_mask: bool = False) -> Dict[str, RuleResult]:
        validator = BaseValidator(self.series, self.rules, return_mask=return_mask)
        return validator.run()

@dataclass
class DateValidator:
    series: pd.Series
    rules: List[ValidationRule] = field(default_factory=list)

    def __post_init__(self):
        if not self.rules:
            self.rules = [IsDateRule()]

    def add_rule(self, rule: ValidationRule):
        self.rules.append(rule)

    def run(self, return_mask: bool = False) -> Dict[str, RuleResult]:
        validator = BaseValidator(self.series, self.rules, return_mask=return_mask)
        return validator.run()

@dataclass
class CategoryValidator:
    series: pd.Series
    rules: List[ValidationRule] = field(default_factory=list)

    def __post_init__(self):
        if not self.rules:
            self.rules = [IsCategoryRule()]

    def add_rule(self, rule: ValidationRule):
        self.rules.append(rule)

    def run(self, return_mask: bool = False) -> Dict[str, RuleResult]:
        validator = BaseValidator(self.series, self.rules, return_mask=return_mask)
        return validator.run()

# -------------------------
# Plugin loader
# -------------------------
def load_rules_from_directory(path: Union[str, Path]) -> None:
    path = Path(path)
    if not path.exists() or not path.is_dir():
        raise FileNotFoundError(path)
    for file in path.glob("*.py"):
        spec = importlib.util.spec_from_file_location(file.stem, file)
        mod = importlib.util.module_from_spec(spec)
        try:
            spec.loader.exec_module(mod)  # type: ignore
            # Discover ValidationRule subclasses and register
            for _, obj in inspect.getmembers(mod):
                if inspect.isclass(obj) and issubclass(obj, ValidationRule) and (obj is not ValidationRule):
                    _RULE_REGISTRY[obj.__name__] = obj
                    logger.info("Loaded plugin rule: %s from %s", obj.__name__, file)
        except Exception:
            logger.exception("Failed to load plugin module %s", file)

# -------------------------
# Small test / example usage
# -------------------------
if __name__ == "__main__":
    # Example data
    s_num = pd.Series([1, 2.0, "3", "x", np.nan, 5.123, 7], index=list(range(7)))
    print("Series numeric sample:\n", s_num)

    nv = NumberValidator(s_num)
    # Add exact decimal rule (2 dp) and range
    nv.add_rule(ExactDecimalPlacesRule(dp=3, use_original_strings=False))
    nv.add_rule(InRangeRule(low=0, high=10, inclusive=True))
    res_num = nv.run(return_mask=False)
    print("\nNumberValidator results (indexes):")
    for k, idx in res_num.items():
        print(f"{k}: {list(idx)}")

    # String examples
    s_str = pd.Series(["abc", "def", "ghi", "abcde", np.nan, "1.23", "1.230"], index=list(range(7)))
    sv = StringValidator(s_str)
    sv.add_rule(InLengthRangeRule(min_len=1, max_len=4))
    sv.add_rule(SubstringExistsRule(substring="1.23", na_fail=False))
    res_str = sv.run(return_mask=False)
    print("\nStringValidator results:")
    for k, idx in res_str.items():
        print(f"{k}: {list(idx)}")

    # Date example
    s_date = pd.Series(["2020-01-01", "notadate", "2021-05-05", np.nan])
    dv = DateValidator(s_date)
    dv.add_rule(DateInRangeRule(low="2020-01-01", high="2020-12-31"))
    print("\nDateValidator results:", dv.run())

    # UUID example
    s_uuid = pd.Series([str(uuid.uuid4()), "not-uuid", np.nan])
    uv = BaseValidator(s_uuid, [UUIDRule()])
    print("\nUUID rule result:", uv.run())

    # Plugin loader usage (create plugin files in a folder then load)
    # load_rules_from_directory("plugins")
